# 📊 Data Analyst Candidate Selection Tool
### Interactive Edition

Upload your Excel file, explore candidates, and visualise insights — all without editing any code.

---


## ⚙️ Step 1 — Setup

In [ ]:
# Install required packages (run once)
import sys
!{sys.executable} -m pip install ipywidgets openpyxl xlrd -q

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from datetime import datetime
import io

print("✅ All packages loaded successfully!")


## 📂 Step 2 — Upload Your Excel File

In [ ]:
# ---------- state ----------
state = {"df": None, "filtered": None}

# ---------- upload widget ----------
upload_widget = widgets.FileUpload(
    accept=".xlsx,.xls",
    multiple=False,
    description="📁 Upload Excel",
    button_style="info",
    layout=widgets.Layout(width="220px")
)

status_label = widgets.Label(value="⬆️  No file uploaded yet.")

upload_box = widgets.VBox(
    [widgets.HTML("<b>Select your candidate Excel file:</b>"), upload_widget, status_label],
    layout=widgets.Layout(padding="12px", border="1px solid #ddd", border_radius="8px", width="420px")
)

def on_upload(change):
    if not upload_widget.value:
        return
    file_info  = list(upload_widget.value.values())[0]
    file_bytes = file_info["content"]
    try:
        df = pd.read_excel(io.BytesIO(file_bytes))
        state["df"] = df
        status_label.value = f"✅ Loaded  '{file_info['metadata']['name']}'  —  {len(df)} rows, {len(df.columns)} columns"
    except Exception as e:
        status_label.value = f"❌ Error: {e}"

upload_widget.observe(on_upload, names="value")
display(upload_box)


## 👀 Step 3 — Preview Raw Data

In [ ]:
rows_slider = widgets.IntSlider(
    value=5, min=1, max=50, step=1,
    description="Rows:",
    continuous_update=False,
    style={"description_width": "60px"},
    layout=widgets.Layout(width="380px")
)

preview_btn = widgets.Button(
    description="Show Preview",
    button_style="primary",
    icon="table",
    layout=widgets.Layout(width="160px")
)

preview_out = widgets.Output()

def show_preview(_):
    with preview_out:
        clear_output(wait=True)
        if state["df"] is None:
            print("⚠️  Please upload a file first (Step 2).")
            return
        display(HTML("<h4 style='margin:4px 0'>Raw Data Preview</h4>"))
        display(state["df"].head(rows_slider.value))
        print(f"\nShape: {state['df'].shape[0]} rows × {state['df'].shape[1]} columns")
        print(f"Columns: {list(state['df'].columns)}")

preview_btn.on_click(show_preview)
display(widgets.VBox([widgets.HBox([rows_slider, preview_btn]), preview_out]))


## 🔍 Step 4 — Filter Candidates Interactively

In [ ]:
keyword_input = widgets.Text(
    value="Data Analysis",
    placeholder="e.g. Data Analysis",
    description="Skill keyword:",
    style={"description_width": "120px"},
    layout=widgets.Layout(width="380px")
)

top_n_slider = widgets.IntSlider(
    value=5, min=1, max=20, step=1,
    description="Top N:",
    continuous_update=False,
    style={"description_width": "60px"},
    layout=widgets.Layout(width="300px")
)

col_select = widgets.SelectMultiple(
    options=[],               # populated dynamically
    description="Columns:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="360px", height="130px")
)

filter_btn = widgets.Button(
    description="Apply Filter",
    button_style="success",
    icon="filter",
    layout=widgets.Layout(width="160px")
)

filter_out = widgets.Output()

def apply_filter(_):
    with filter_out:
        clear_output(wait=True)
        if state["df"] is None:
            print("⚠️  Please upload a file first.")
            return
        df = state["df"]
        keyword = keyword_input.value.strip()
        top_n   = top_n_slider.value

        # Detect skill column
        skill_col = next((c for c in df.columns if "skill" in c.lower()), None)
        if skill_col is None:
            print(f"❌ No 'Skill' column found. Available columns: {list(df.columns)}")
            return

        # Update column selector
        col_select.options = list(df.columns)
        if not col_select.value:
            # default columns
            defaults = [c for c in ['Name','Mobile Number', skill_col,'Age'] if c in df.columns]
            col_select.value = defaults

        filtered = df[df[skill_col].str.contains(keyword, case=False, na=False)]
        cols = list(col_select.value) if col_select.value else list(df.columns)
        cols = [c for c in cols if c in filtered.columns]
        state["filtered"] = filtered[cols].head(top_n).reset_index(drop=True)

        display(HTML(f"<h4 style='margin:4px 0'>Top {top_n} candidates matching <em>'{keyword}'</em></h4>"))
        display(state["filtered"])
        print(f"\n{len(filtered)} total matches found.")

# auto-populate col_select when df is loaded
def refresh_cols(_=None):
    if state["df"] is not None:
        col_select.options = list(state["df"].columns)

filter_btn.on_click(apply_filter)

display(widgets.VBox([
    widgets.HTML("<b>Filter settings</b>"),
    keyword_input,
    widgets.HBox([top_n_slider]),
    col_select,
    filter_btn,
    filter_out
], layout=widgets.Layout(padding="12px", border="1px solid #ddd", border_radius="8px")))


## 💾 Step 5 — Save Filtered Results

In [ ]:
filename_input = widgets.Text(
    value="filtered_candidates",
    description="Filename:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="340px")
)

save_btn = widgets.Button(
    description="Save to Excel",
    button_style="warning",
    icon="download",
    layout=widgets.Layout(width="170px")
)

save_out = widgets.Output()

def save_results(_):
    with save_out:
        clear_output(wait=True)
        if state["filtered"] is None or state["filtered"].empty:
            print("⚠️  No filtered data. Run Step 4 first.")
            return
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        name = filename_input.value.strip() or "filtered_candidates"
        fname = f"{name}_{timestamp}.xlsx"
        state["filtered"].to_excel(fname, index=False)
        print(f"✅ Saved to: {fname}")

save_btn.on_click(save_results)
display(widgets.VBox([widgets.HBox([filename_input, save_btn]), save_out]))


## 📈 Step 6 — Visualise

In [ ]:
chart_type   = widgets.ToggleButtons(
    options=["Bar Chart", "Pie Chart", "Both"],
    value="Bar Chart",
    description="Chart:",
    button_style="info",
    style={"description_width": "60px"}
)

color_picker = widgets.Dropdown(
    options=["steelblue", "coral", "mediumseagreen", "mediumpurple", "goldenrod"],
    value="steelblue",
    description="Color:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="220px")
)

plot_btn = widgets.Button(
    description="Generate Chart",
    button_style="primary",
    icon="bar-chart",
    layout=widgets.Layout(width="180px")
)

plot_out = widgets.Output()

SKILL_LEVELS = {"Beginner": 1, "Intermediate": 2, "Advanced": 3, "Expert": 4}

def get_skill_level(s):
    parts = str(s).split(" - ")
    return SKILL_LEVELS.get(parts[1].strip(), 0) if len(parts) > 1 else 0

def draw_bar(df, skill_col, color, ax):
    df2 = df.copy()
    df2["_level"] = df2[skill_col].apply(get_skill_level)
    name_col = next((c for c in df2.columns if "name" in c.lower()), df2.columns[0])
    grouped = df2.groupby(name_col)["_level"].sum().sort_values(ascending=False)
    grouped.plot(kind="bar", ax=ax, color=color, edgecolor="white", linewidth=0.8)
    ax.set_title("Candidate Skill Levels", fontsize=14, fontweight="bold", pad=12)
    ax.set_xlabel("Candidate", fontsize=11)
    ax.set_ylabel("Skill Score", fontsize=11)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
    ax.spines[["top","right"]].set_visible(False)
    ax.grid(axis="y", linestyle="--", alpha=0.4)

def draw_pie(df, skill_col, ax):
    counts = df[skill_col].value_counts()
    colors = plt.cm.Set3.colors[:len(counts)]
    wedges, texts, autotexts = ax.pie(
        counts, labels=None, autopct="%1.1f%%",
        startangle=140, colors=colors,
        pctdistance=0.82, wedgeprops=dict(linewidth=1.4, edgecolor="white")
    )
    for at in autotexts:
        at.set_fontsize(9)
    ax.legend(
        wedges, counts.index, title="Skills",
        loc="center left", bbox_to_anchor=(1, 0.5), fontsize=9
    )
    ax.set_title("Skill Distribution", fontsize=14, fontweight="bold")

def generate_charts(_):
    with plot_out:
        clear_output(wait=True)
        df = state["filtered"] if state["filtered"] is not None else state["df"]
        if df is None:
            print("⚠️  No data available. Upload a file and apply a filter first.")
            return

        skill_col = next((c for c in df.columns if "skill" in c.lower()), None)
        if skill_col is None:
            print(f"❌ No skill column found. Columns: {list(df.columns)}")
            return

        choice = chart_type.value
        color  = color_picker.value

        if choice == "Both":
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            draw_bar(df, skill_col, color, ax1)
            draw_pie(df, skill_col, ax2)
        elif choice == "Bar Chart":
            fig, ax = plt.subplots(figsize=(9, 5))
            draw_bar(df, skill_col, color, ax)
        else:
            fig, ax = plt.subplots(figsize=(8, 6))
            draw_pie(df, skill_col, ax)

        plt.tight_layout()
        plt.show()

plot_btn.on_click(generate_charts)

display(widgets.VBox([
    widgets.HTML("<b>Chart options</b>"),
    widgets.HBox([chart_type]),
    widgets.HBox([color_picker]),
    plot_btn,
    plot_out
], layout=widgets.Layout(padding="12px", border="1px solid #ddd", border_radius="8px")))


## 📋 Step 7 — Quick Summary Statistics

In [ ]:
summary_btn = widgets.Button(
    description="Show Summary",
    button_style="info",
    icon="info-circle",
    layout=widgets.Layout(width="180px")
)

summary_out = widgets.Output()

def show_summary(_):
    with summary_out:
        clear_output(wait=True)
        df = state["filtered"] if state["filtered"] is not None else state["df"]
        if df is None:
            print("⚠️  No data available.")
            return
        
        display(HTML("<h4 style='margin:4px 0'>Dataset Summary</h4>"))
        print(f"Total records   : {len(df)}")
        
        age_col  = next((c for c in df.columns if "age" in c.lower()),  None)
        skill_col= next((c for c in df.columns if "skill" in c.lower()),None)
        name_col = next((c for c in df.columns if "name" in c.lower()), None)
        
        if age_col:
            ages = pd.to_numeric(df[age_col], errors="coerce").dropna()
            if not ages.empty:
                print(f"Age — avg: {ages.mean():.1f}, min: {int(ages.min())}, max: {int(ages.max())}")
        
        if skill_col:
            print(f"\nSkill breakdown:")
            display(df[skill_col].value_counts().rename("Count").to_frame())
        
        if name_col:
            print(f"\nAll candidate names: {', '.join(df[name_col].dropna().astype(str).tolist())}")

summary_btn.on_click(show_summary)
display(widgets.VBox([summary_btn, summary_out]))


---
## 🔄 Reset Everything

In [ ]:
reset_btn = widgets.Button(
    description="Reset Session",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="180px")
)

def reset(_):
    state["df"] = None
    state["filtered"] = None
    print("✅ Session reset. Re-run Step 2 to start fresh.")

reset_btn.on_click(reset)
display(reset_btn)
